# Chương 6. Hồi quy Logistic và Phân loại nhị phân
*Giáo trình **Học máy – Cơ sở và Ứng dụng với Python*** · Notebook thực hành

Chạy lần lượt từng ô (**Runtime → Run all**). Không cần GPU và không cần tải dữ liệu ngoài (chỉ dùng dữ liệu có sẵn trong `scikit-learn`).

## Mục tiêu học tập
1. Giải thích vì sao không nên dùng hồi quy tuyến tính trực tiếp cho phân loại.
2. Hiểu hàm **sigmoid** và cách hồi quy Logistic mô hình hoá xác suất.
3. Hiểu hàm mất mát **Binary Cross-Entropy (Log Loss)** và vì sao tốt hơn MSE.
4. Hiểu **ngưỡng quyết định** và **ranh giới quyết định**.
5. Tính và diễn giải **ma trận nhầm lẫn**, Precision, Recall, F1-score.
6. Cài đặt bằng scikit-learn, vẽ đường cong **ROC** và tính **AUC**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (confusion_matrix, classification_report, ConfusionMatrixDisplay,
                             RocCurveDisplay, roc_auc_score, accuracy_score,
                             precision_score, recall_score, f1_score)

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)
print("Đã sẵn sàng!")

## 1. Vì sao không dùng hồi quy tuyến tính cho phân loại?

Với nhãn $y\in\{0,1\}$, hồi quy tuyến tính cho $\hat y$ là số thực bất kỳ (có thể $<0$ hoặc $>1$) — không diễn giải được như xác suất, và đường thẳng rất nhạy với điểm ở xa. Ta thử minh hoạ: một điểm "xa" kéo đường thẳng làm lệch ranh giới.

In [ ]:
rng = np.random.default_rng(1)
x0 = rng.normal(2, 0.8, 40)      # lớp 0
x1 = rng.normal(5, 0.8, 40)      # lớp 1
x = np.r_[x0, x1]; y = np.r_[np.zeros(40), np.ones(40)]

x_ngoai = np.r_[x, 25.0]; y_ngoai = np.r_[y, 1.0]   # thêm một điểm lớp 1 ở rất xa

def duong_thang(xv, yv):
    m = LinearRegression().fit(xv.reshape(-1, 1), yv)
    return m

m1 = duong_thang(x, y); m2 = duong_thang(x_ngoai, y_ngoai)
xs = np.linspace(0, 26, 200).reshape(-1, 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for a, (xv, yv, m, tt) in zip(ax, [(x, y, m1, "Không có điểm xa"), (x_ngoai, y_ngoai, m2, "Thêm 1 điểm ở x = 25")]):
    a.scatter(xv, yv, c=yv, cmap="coolwarm", edgecolor="k", alpha=0.8)
    a.plot(xs, m.predict(xs), color="black", lw=2, label="Hồi quy tuyến tính")
    a.axhline(0.5, color="gray", ls="--")
    # điểm mà đường thẳng cắt y = 0.5
    ranh = (0.5 - m.intercept_) / m.coef_[0]
    a.axvline(ranh, color="green", ls=":", label=f"Ranh giới ≈ {ranh:.2f}")
    a.set_title(tt); a.set_xlabel("x"); a.legend()
ax[0].set_ylabel("y / ŷ")
plt.tight_layout(); plt.show()

Quan sát: chỉ vì một điểm thuộc lớp 1 nằm xa (đúng phía), ranh giới bị **dịch chuyển** và nhiều điểm bị phân loại sai; ngoài ra $\hat y$ vượt quá $[0,1]$.

## 2. Hàm sigmoid

$$\sigma(z)=\frac{1}{1+e^{-z}},\qquad z=w_0+w_1x_1+\dots+w_nx_n,\qquad \hat p=\sigma(z)=P(y=1\mid\mathbf{x})$$

Tính chất: $\sigma(0)=0{,}5$; $z\to+\infty\Rightarrow\sigma\to1$; $z\to-\infty\Rightarrow\sigma\to0$.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-10, -2, 0, 2, 10]:
    print(f"σ({z:>3}) = {sigmoid(z):.5f}")

z = np.linspace(-8, 8, 300)
plt.plot(z, sigmoid(z), lw=2)
plt.axhline(0.5, color="gray", ls="--"); plt.axvline(0, color="gray", ls="--")
plt.xlabel("z"); plt.ylabel("σ(z)"); plt.title("Hàm sigmoid")
plt.show()

## 3. Hàm mất mát Binary Cross-Entropy (Log Loss)

$$L=-\frac1m\sum_i\Big[y_i\log\hat p_i+(1-y_i)\log(1-\hat p_i)\Big]$$

- Nếu $y=1$: mất mát $=-\log\hat p$. Dự đoán đúng ($\hat p\to1$) → $\approx0$; sai hoàn toàn ($\hat p\to0$) → $\to\infty$.
- Nếu $y=0$: mất mát $=-\log(1-\hat p)$.

Hàm này **lồi** theo $\mathbf w$ nên Gradient Descent hội tụ tới cực tiểu toàn cục, trong khi MSE kết hợp sigmoid là **không lồi**.

In [ ]:
p = np.linspace(0.001, 0.999, 400)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(p, -np.log(p), label="y = 1:  −log(p̂)")
ax[0].plot(p, -np.log(1 - p), label="y = 0:  −log(1 − p̂)")
ax[0].set_xlabel("Xác suất dự đoán p̂"); ax[0].set_ylabel("Mất mát"); ax[0].set_ylim(0, 6)
ax[0].set_title("Binary Cross-Entropy cho một mẫu"); ax[0].legend()

# So sánh theo trọng số w trên một bài toán 1 chiều nhỏ: MSE (không lồi) vs BCE (lồi)
rng = np.random.default_rng(3)
xd = np.r_[rng.normal(-1, 1, 30), rng.normal(1, 1, 30)]
yd = np.r_[np.zeros(30), np.ones(30)]
ws = np.linspace(-6, 10, 300)
bce = [-np.mean(yd*np.log(sigmoid(w*xd)+1e-12) + (1-yd)*np.log(1-sigmoid(w*xd)+1e-12)) for w in ws]
mse = [np.mean((yd - sigmoid(w*xd))**2) for w in ws]
ax[1].plot(ws, bce / max(bce), label="BCE (chuẩn hoá)")
ax[1].plot(ws, mse / max(mse), label="MSE (chuẩn hoá)")
ax[1].set_xlabel("w"); ax[1].set_ylabel("Mất mát tương đối")
ax[1].set_title("Mặt mất mát theo w"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Tự cài đặt hồi quy Logistic bằng Gradient Descent

Gradient của Log Loss có dạng gọn: $\nabla_{\mathbf w}L=\frac1m X^\top(\hat{\mathbf p}-\mathbf y)$. Ta cài đặt và so sánh với scikit-learn trên bộ dữ liệu Breast Cancer (đã chuẩn hoá).

In [ ]:
du_lieu = load_breast_cancer()
X, y = du_lieu.data, du_lieu.target
# Quy ước: 0 = ác tính (malignant), 1 = lành tính (benign)
print("Số mẫu:", X.shape[0], "| Số đặc trưng:", X.shape[1])
print("Phân bố lớp:", dict(zip(*np.unique(y, return_counts=True))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

bo_chuan_hoa = StandardScaler()
X_train_ch = bo_chuan_hoa.fit_transform(X_train)
X_test_ch = bo_chuan_hoa.transform(X_test)

In [ ]:
def logistic_gd(X, y, alpha=0.1, so_lan=300):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    lich_su = []
    for _ in range(so_lan):
        p = sigmoid(Xb @ w)
        mat_mat = -np.mean(y*np.log(p + 1e-12) + (1-y)*np.log(1-p + 1e-12))
        lich_su.append(mat_mat)
        w -= alpha * Xb.T @ (p - y) / len(y)
    return w, lich_su

w_gd, ls = logistic_gd(X_train_ch, y_train)
p_gd = sigmoid(np.c_[np.ones(len(X_test_ch)), X_test_ch] @ w_gd)
y_gd = (p_gd >= 0.5).astype(int)
print(f"Accuracy (GD tự cài) = {accuracy_score(y_test, y_gd):.4f}")

plt.plot(ls); plt.xlabel("Vòng lặp"); plt.ylabel("Log Loss"); plt.title("Đường cong mất mát")
plt.show()

## 5. Ví dụ chính: phân loại khối u vú bằng scikit-learn

In [ ]:
mo_hinh = LogisticRegression(max_iter=1000)
mo_hinh.fit(X_train_ch, y_train)

y_du_doan = mo_hinh.predict(X_test_ch)
xac_suat = mo_hinh.predict_proba(X_test_ch)[:, 1]        # P(y = 1) = P(lành tính)

ma_tran = confusion_matrix(y_test, y_du_doan)
print("Ma trận nhầm lẫn:")
print(ma_tran)
print()
print(classification_report(y_test, y_du_doan, target_names=du_lieu.target_names))
auc = roc_auc_score(y_test, xac_suat)
print(f"AUC = {auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(ma_tran, display_labels=du_lieu.target_names).plot(
    ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Ma trận nhầm lẫn"); axes[0].grid(False)

RocCurveDisplay.from_predictions(y_test, xac_suat, ax=axes[1])
axes[1].plot([0, 1], [0, 1], ls="--", color="gray", label="Phân loại ngẫu nhiên")
axes[1].set_title("Đường cong ROC"); axes[1].legend()
plt.tight_layout(); plt.show()

**Đọc kết quả.** Với nhãn `0 = malignant` (ác tính) và `1 = benign` (lành tính), `confusion_matrix` của scikit-learn có hàng = nhãn thật, cột = nhãn dự đoán:

| | Dự đoán: ác tính (0) | Dự đoán: lành tính (1) |
|---|---|---|
| **Thật: ác tính (0)** | đúng | bỏ sót ca ác tính |
| **Thật: lành tính (1)** | báo động giả | đúng |

Trong y khoa, **bỏ sót ca ác tính** (dự đoán nhầm thành lành tính) nguy hiểm hơn báo động giả. AUC gần 1 cho thấy mô hình phân tách hai lớp rất tốt trên toàn dải ngưỡng, không chỉ ở ngưỡng 0,5.

### Tính tay các độ đo từ ma trận nhầm lẫn
Coi **lớp ác tính (0)** là lớp "dương" cần phát hiện:
$$\text{Precision}=\frac{TP}{TP+FP},\quad \text{Recall}=\frac{TP}{TP+FN},\quad F_1=\frac{2PR}{P+R}$$

In [ ]:
# Với lớp dương = ác tính (nhãn 0): TP = thật 0 & đoán 0
TP = ma_tran[0, 0]; FN = ma_tran[0, 1]
FP = ma_tran[1, 0]; TN = ma_tran[1, 1]
precision = TP / (TP + FP); recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
acc = (TP + TN) / ma_tran.sum()
print(f"TP={TP}, FN={FN}, FP={FP}, TN={TN}")
print(f"Accuracy={acc:.4f}  Precision={precision:.4f}  Recall={recall:.4f}  F1={f1:.4f}")

# Đối chiếu với scikit-learn (pos_label=0)
assert np.isclose(precision, precision_score(y_test, y_du_doan, pos_label=0))
assert np.isclose(recall, recall_score(y_test, y_du_doan, pos_label=0))
assert np.isclose(f1, f1_score(y_test, y_du_doan, pos_label=0))
print("✅ Khớp với scikit-learn")

## 6. Ranh giới quyết định (2 đặc trưng)
Ranh giới là tập điểm có $\hat p=0{,}5\iff z=0$: một **đường thẳng** → hồi quy Logistic là bộ phân loại tuyến tính.

In [ ]:
i1, i2 = list(du_lieu.feature_names).index("mean radius"), list(du_lieu.feature_names).index("mean texture")
X2 = StandardScaler().fit_transform(X[:, [i1, i2]])
m2d = LogisticRegression().fit(X2, y)

gx, gy = np.meshgrid(np.linspace(-3, 4, 300), np.linspace(-3, 4.5, 300))
pg = m2d.predict_proba(np.c_[gx.ravel(), gy.ravel()])[:, 1].reshape(gx.shape)

plt.contourf(gx, gy, pg, levels=20, cmap="coolwarm", alpha=0.35)
plt.contour(gx, gy, pg, levels=[0.5], colors="k", linewidths=2)
plt.scatter(X2[:, 0], X2[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=20)
plt.xlabel("mean radius (chuẩn hoá)"); plt.ylabel("mean texture (chuẩn hoá)")
plt.title("Ranh giới quyết định p̂ = 0,5 (đường đen)")
plt.show()

## 7. Bài tập lập trình (có gợi ý)

Mỗi bài: tự làm vào ô code, kẹt thì mở gợi ý, cuối cùng mới xem lời giải.

### Bài 1 – Không chuẩn hoá dữ liệu
Huấn luyện lại mô hình **không** dùng `StandardScaler`. So sánh cảnh báo `ConvergenceWarning`, Accuracy và AUC với bản có chuẩn hoá.

<details>
<summary>💡 Gợi ý 1</summary>

Dùng trực tiếp `X_train`, `X_test` (chưa chuẩn hoá) và `LogisticRegression(max_iter=1000)`.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

Muốn biết số vòng lặp thực tế, xem thuộc tính `mo_hinh.n_iter_` sau khi huấn luyện. Để thấy cảnh báo, thử `max_iter=100` hoặc dùng `solver='lbfgs'` mặc định.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

Tính `accuracy_score(y_test, mh.predict(...))` và `roc_auc_score(y_test, mh.predict_proba(...)[:,1])` cho cả hai mô hình rồi đặt vào một bảng `pd.DataFrame`.

</details>

In [ ]:
# TODO: Bài 1
...

<details>
<summary>✅ Lời giải Bài 1</summary>

```python
import warnings
rows = []
for ten, (Xtr, Xte) in {"Có chuẩn hoá": (X_train_ch, X_test_ch), "Không chuẩn hoá": (X_train, X_test)}.items():
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        mh = LogisticRegression(max_iter=100).fit(Xtr, y_train)
        canh_bao = any("ConvergenceWarning" in str(x.category) for x in w)
    rows.append({"Mô hình": ten,
                 "Số vòng lặp": int(mh.n_iter_[0]),
                 "Cảnh báo hội tụ": canh_bao,
                 "Accuracy": accuracy_score(y_test, mh.predict(Xte)),
                 "AUC": roc_auc_score(y_test, mh.predict_proba(Xte)[:, 1])})
bai1 = pd.DataFrame(rows).round(4)
print(bai1)
```

**Nhận xét:** khi đặc trưng có thang đo rất khác nhau, tối ưu hoá hội tụ chậm hơn (cần nhiều vòng hơn, dễ cảnh báo) và chất lượng thường giảm nhẹ; chuẩn hoá giúp hội tụ nhanh và ổn định.

</details>

### Bài 2 – Quét ngưỡng quyết định
Với các ngưỡng 0.1, 0.3, 0.5, 0.7, 0.9, áp ngưỡng **thủ công** lên `xac_suat` (xác suất lớp 1). Tính và in Precision, Recall, F1; vẽ Precision và Recall theo ngưỡng.

<details>
<summary>💡 Gợi ý 1</summary>

Dự đoán nhãn bằng ngưỡng: `y_t = (xac_suat >= nguong).astype(int)`.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

Dùng `precision_score(y_test, y_t, zero_division=0)`, tương tự cho recall và f1 (ở đây lớp dương là nhãn 1 = lành tính).

</details>

<details>
<summary>💡 Gợi ý 3</summary>

Lưu kết quả mỗi ngưỡng vào list, tạo `DataFrame`, rồi `plt.plot(df['nguong'], df['precision'])` và tương tự cho recall.

</details>

<details>
<summary>💡 Gợi ý 4</summary>

Khi tăng ngưỡng, mô hình khó dự đoán lớp 1 hơn ⇒ Precision thường tăng, Recall giảm.

</details>

In [ ]:
# TODO: Bài 2
...

<details>
<summary>✅ Lời giải Bài 2</summary>

```python
ds = []
for nguong in [0.1, 0.3, 0.5, 0.7, 0.9]:
    y_t = (xac_suat >= nguong).astype(int)
    ds.append({"nguong": nguong,
               "precision": precision_score(y_test, y_t, zero_division=0),
               "recall": recall_score(y_test, y_t),
               "f1": f1_score(y_test, y_t)})
bai2 = pd.DataFrame(ds)
print(bai2.round(4).to_string(index=False))

plt.plot(bai2["nguong"], bai2["precision"], "o-", label="Precision")
plt.plot(bai2["nguong"], bai2["recall"], "s-", label="Recall")
plt.xlabel("Ngưỡng quyết định"); plt.ylabel("Giá trị"); plt.title("Precision và Recall theo ngưỡng")
plt.legend(); plt.show()
```

</details>

### Bài 3 (nâng cao) – Dữ liệu mất cân bằng
Tạo dữ liệu bằng `make_classification(weights=[0.95, 0.05])`. So sánh hồi quy Logistic mặc định và `class_weight="balanced"`: ma trận nhầm lẫn và F1 của lớp thiểu số (lớp 1).

<details>
<summary>💡 Gợi ý 1</summary>

`make_classification(n_samples=5000, n_features=10, n_informative=5, weights=[0.95, 0.05], flip_y=0.01, class_sep=1.2, random_state=42)`.

</details>

<details>
<summary>💡 Gợi ý 2</summary>

Nhớ dùng `stratify=y` khi chia train/test để giữ tỉ lệ 95/5.

</details>

<details>
<summary>💡 Gợi ý 3</summary>

Huấn luyện hai mô hình: `LogisticRegression(max_iter=1000)` và `LogisticRegression(max_iter=1000, class_weight='balanced')`.

</details>

<details>
<summary>💡 Gợi ý 4</summary>

F1 của lớp thiểu số: `f1_score(y_te, y_dd, pos_label=1)`. Kỳ vọng: bản `balanced` tăng mạnh Recall lớp 1 nhưng giảm Precision; F1 có thể không tăng.

</details>

In [ ]:
# TODO: Bài 3
...

<details>
<summary>✅ Lời giải Bài 3</summary>

```python
Xi, yi = make_classification(n_samples=5000, n_features=10, n_informative=5, weights=[0.95, 0.05],
                             flip_y=0.01, class_sep=1.2, random_state=42)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.25, random_state=42, stratify=yi)
print("Tỉ lệ lớp 1:", yi.mean().round(3))

ket_qua = {}
for ten, cw in {"Mặc định": None, "balanced": "balanced"}.items():
    mh = LogisticRegression(max_iter=1000, class_weight=cw).fit(Xi_tr, yi_tr)
    dd = mh.predict(Xi_te)
    ket_qua[ten] = {"cm": confusion_matrix(yi_te, dd),
                    "f1": f1_score(yi_te, dd, pos_label=1),
                    "recall": recall_score(yi_te, dd, pos_label=1),
                    "precision": precision_score(yi_te, dd, pos_label=1, zero_division=0)}
    print(f"\n[{ten}] F1 lớp 1 = {ket_qua[ten]['f1']:.3f} | Recall = {ket_qua[ten]['recall']:.3f} | Precision = {ket_qua[ten]['precision']:.3f}")
    print(ket_qua[ten]["cm"])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, (ten, r) in zip(ax, ket_qua.items()):
    ConfusionMatrixDisplay(r["cm"]).plot(ax=a, cmap="Blues", colorbar=False)
    a.set_title(ten); a.grid(False)
plt.tight_layout(); plt.show()
```

**Nhận xét:** `class_weight='balanced'` phạt nặng hơn việc sai ở lớp thiểu số nên Recall lớp 1 tăng mạnh (bắt được nhiều ca hiếm hơn), đổi lại Precision giảm nhiều vì có thêm báo động giả. F1 có thể chỉ thay đổi ít hoặc thậm chí không tăng — tuỳ bài toán, hãy chọn cấu hình theo chi phí của bỏ sót so với báo động giả (hoặc kết hợp chỉnh ngưỡng).

</details>

## 8. Bài tập củng cố (làm vào vở)
**Nhận biết**
1. Viết công thức sigmoid; giá trị khi $z=0$, $z$ rất lớn, $z$ rất nhỏ.
2. Công thức Precision và Recall; ý nghĩa mỗi độ đo.

**Thông hiểu**

3. Vì sao không dùng MSE cho hồi quy Logistic?
4. Mô hình phát hiện gian lận có Accuracy 99,5% nhưng Recall 40% (gian lận chỉ ~0,5%): vì sao Accuracy gây hiểu lầm?
5. Hạ ngưỡng từ 0,5 xuống 0,2 thì Precision, Recall thay đổi thế nào?

**Vận dụng**

6. Lọc email spam và sàng lọc hành khách nguy hiểm: ưu tiên Precision hay Recall? Chỉnh ngưỡng ra sao?

## 9. Tóm tắt
- Hồi quy Logistic: $\hat p=\sigma(\mathbf w^\top\mathbf x)$, thuộc nhóm bộ phân loại tuyến tính.
- Log Loss là hàm mất mát lồi, phạt nặng dự đoán sai tự tin.
- Ngưỡng 0,5 chỉ là mặc định — chọn theo chi phí FP/FN của bài toán.
- Dùng ma trận nhầm lẫn, Precision/Recall/F1 và ROC–AUC; cẩn trọng với Accuracy khi dữ liệu mất cân bằng.
- **Chuẩn hoá đặc trưng** giúp tối ưu hội tụ nhanh và ổn định.

➡️ Chương 7: thuật toán K-Láng giềng gần nhất (K-NN).